In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
print(df.columns.tolist())

['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'year_added', 'month_added', 'duration_num', 'duration_type', 'type_encoded']


In [4]:
# Dataset load karein
df = pd.read_csv('netflix_cleaned.csv')

# Sirf zaroori columns rakhein (description nahi hai, isliye listed_in use karenge)
df_rec = df[['title', 'type', 'listed_in']].copy()

# Missing values hata dein
df_rec = df_rec.dropna(subset=['listed_in'])

print("Dataset loaded!")
print("Total titles:", len(df_rec))
df_rec.head()

Dataset loaded!
Total titles: 8790


,title,type,listed_in
0,Dick Johnson Is Dead,Movie,Documentaries
1,Ganglands,TV Show,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,TV Show,"TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies"
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies"


In [5]:
# Genre ko feature bana dein
df_rec['features'] = df_rec['listed_in'].str.lower()

print("Features ban gaye!")
df_rec[['title', 'features']].head()

Features ban gaye!


,title,features
0,Dick Johnson Is Dead,documentaries
1,Ganglands,"crime tv shows, international tv shows, tv act..."
2,Midnight Mass,"tv dramas, tv horror, tv mysteries"
3,Confessions of an Invisible Girl,"children & family movies, comedies"
4,Sankofa,"dramas, independent movies, international movies"


In [6]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_rec['features'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 44)


In [7]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)
print("Similarity calculate ho gayi!")

Cosine Similarity Matrix Shape: (8790, 8790)
Similarity calculate ho gayi!


In [8]:
indices = pd.Series(df_rec.index, index=df_rec['title']).drop_duplicates()

def recommend(title, n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    return df_rec['title'].iloc[movie_indices]

print("Recommendation function ban gaya!")

Recommendation function ban gaya!


In [9]:
sample_title = df_rec['title'].iloc[0]
print(f"Recommendations for '{sample_title}':")
print("="*50)
print(recommend(sample_title, 5))

Recommendations for 'Dick Johnson Is Dead':
80            My Heroes Were Cowboys
110    9to5: The Story of a Movement
115             Why Did You Kill Me?
151      Headspace: Unwind Your Mind
171                    Final Account
Name: title, dtype: str
